In [2]:
import polars as pl
from dotenv import load_dotenv

from prim_client import PrimClient


In [3]:
p_conf = "/home/valentin/work/conf/ratp.env"
load_dotenv(p_conf)

True

In [4]:
# Get perimeter data from PRIM website - pqt format
# https://prim.iledefrance-mobilites.fr/fr/jeux-de-donnees/perimetre-des-donnees-tr-disponibles-plateforme-idfm

In [5]:
p_pqt = "datasets/perimetre-des-donnees-tr-disponibles-plateforme-idfm.parquet"
df = pl.read_parquet(p_pqt)

In [6]:
dtype = pl.Struct(
    {
        "@srsName": pl.String,
        "ns2:Longitude": pl.String,
        "ns2:Latitude": pl.String,
    }
)
df = df.with_columns(pl.col("ns2_location").str.json_decode(dtype))


In [7]:
dtype = pl.Struct({"ns2:LineRef": pl.List(pl.String)})
df = df.with_columns(pl.col("ns2_lines").str.json_decode(dtype))

In [8]:
df = df.unnest("ns2_lines", "ns2_location")

In [9]:
df.head()

line,name_line,ns2_stoppointref,ns2_stopname,ns2:LineRef,@srsName,ns2:Longitude,ns2:Latitude,ns2_location_ns2longitude,operatorname
str,str,str,str,list[str],str,str,str,str,str
"""STIF:Line::C02107:""","""3137""","""STIF:StopPoint:Q:6215:""","""Ãglise""","[""STIF:Line::C02107:"", ""STIF:Line::C02108:"", … ""STIF:Line::C02088:""]","""EPSG:2154""","""6841919.0""","""681371.0""","""6841919.0""","""Keolis Portes et Val de Brie"""
"""STIF:Line::C00115:""","""6593""","""STIF:StopPoint:Q:478867:""",null,"[""STIF:Line::C00111:"", ""STIF:Line::C00115:""]",null,"""0.0""","""0.0""","""0.0""","""Keolis Seine et Oise Est"""
"""STIF:Line::C02470:""","""4411""","""STIF:StopPoint:Q:2759:""","""Gare d'Ãtampes""","[""STIF:Line::C01851:"", ""STIF:Line::C01568:"", … ""STIF:Line::C00982:""]","""EPSG:2154""","""6815469.0""","""637847.0""","""6815469.0""","""Francilité Ouest Essonne"""
"""STIF:Line::C01224:""","""210""","""STIF:StopPoint:Q:479089:""","""Rue du 4eme Zouave""","[""STIF:Line::C01224:""]","""EPSG:2154""","""6859788.0""","""664990.0""","""6859788.0""","""RATP Cap Boucles de Marne"""
"""STIF:Line::C01902:""","""F""","""STIF:StopPoint:Q:27164:""","""La Fontaine""","[""STIF:Line::C01902:""]","""EPSG:2154""","""6845732.0""","""646200.0""","""6845732.0""","""Autocars Dominique"""


In [10]:
dfi = df.filter(
    pl.col("name_line") == "L",
    pl.col("ns2_stopname") == "Saint-Cloud"
)

In [11]:
dfi

line,name_line,ns2_stoppointref,ns2_stopname,ns2:LineRef,@srsName,ns2:Longitude,ns2:Latitude,ns2_location_ns2longitude,operatorname
str,str,str,str,list[str],str,str,str,str,str
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:471421:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860963.0""","""642589.0""","""6860963.0""","""SNCF"""
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:471906:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860960.0""","""642578.0""","""6860960.0""","""SNCF"""
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:471432:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860962.0""","""642574.0""","""6860962.0""","""SNCF"""
"""STIF:Line::C01740:""","""L""","""STIF:StopPoint:Q:470852:""","""Saint-Cloud""","[""STIF:Line::C01741:"", ""STIF:Line::C01736:"", ""STIF:Line::C01740:""]","""EPSG:2154""","""6860962.0""","""642594.0""","""6860962.0""","""SNCF"""


In [12]:
ligne_ref = dfi.get_column("line").to_list()[0]
ligne_ref

'STIF:Line::C01740:'

In [13]:
prim = PrimClient()
data = prim.estimated_timetable()